In [21]:
import pulp
import math
import random

# -----------------------------
# REDUCED CUSTOMER SET
# -----------------------------
N = 13
customers = list(range(1, N+1))
depot = 0
nodes = [depot] + customers

m = 1
vehicles = [2]

speed = 25.23
H = 240

Q = {1: 999}

# demand
q = {j:1 for j in customers}

# service times
s = {}
for j in customers:
    s[j] = random.choice([12.7,13.4,21.6])

# -----------------------------
# coordinates (5–7 mile radius)
# -----------------------------
coords = {0:(0,0)}

for j in customers:
    angle = 2*math.pi*j/N
    radius = random.uniform(5,7)
    coords[j] = (
        radius*math.cos(angle),
        radius*math.sin(angle)
    )

# distance matrix
d = {}
for i in nodes:
    for j in nodes:
        if i != j:
            x1,y1 = coords[i]
            x2,y2 = coords[j]
            d[i,j] = math.sqrt((x1-x2)**2 + (y1-y2)**2)

# -----------------------------
# MODEL
# -----------------------------
model = pulp.LpProblem("VRP13", pulp.LpMinimize)

x = pulp.LpVariable.dicts(
    "x",
    [(i,j) for i in nodes for j in nodes if i!=j],
    cat="Binary"
)

u = pulp.LpVariable.dicts(
    "u",
    customers,
    lowBound=1,
    upBound=N,
    cat="Continuous"
)

# objective
model += pulp.lpSum(
    d[i,j]*x[i,j]
    for i in nodes for j in nodes if i!=j
)

# visit once
for j in customers:
    model += pulp.lpSum(x[i,j] for i in nodes if i!=j) == 1
    model += pulp.lpSum(x[j,k] for k in nodes if j!=k) == 1

# start depot
model += pulp.lpSum(x[0,j] for j in customers) == 1

# return depot
model += pulp.lpSum(x[i,0] for i in customers) == 1

# MTZ subtour elimination
for i in customers:
    for j in customers:
        if i != j:
            model += u[i] - u[j] + N*x[i,j] <= N-1

# shift constraint
travel_time = pulp.lpSum(
    (d[i,j]/speed)*60*x[i,j]
    for i in nodes for j in nodes if i!=j
)

service_time = sum(s[j] for j in customers)

model += travel_time + service_time <= H

# solve
solver = pulp.PULP_CBC_CMD(msg=True)
model.solve(solver)

print("Status:", pulp.LpStatus[model.status])

# -----------------------------
# Route reconstruction
# -----------------------------
route = [0]
current = 0

while True:
    next_node = None
    for j in nodes:
        if current != j and pulp.value(x[current,j]) > 0.5:
            next_node = j
            break

    if next_node is None:
        break

    route.append(next_node)

    if next_node == 0:
        break

    current = next_node

print("Optimal Route:", route)

distance_total = sum(
    d[route[i],route[i+1]]
    for i in range(len(route)-1)
)

travel_total = (distance_total/speed)*60
service_total = sum(s[j] for j in customers)

print("Distance:", round(distance_total,2),"miles")
print("Travel Time:", round(travel_total,2),"min")
print("Service Time:", round(service_total,2),"min")
print("Total Time:", round(travel_total+service_total,2),"min")

Status: Infeasible
Optimal Route: [0, 3, 2]
Distance: 9.49 miles
Travel Time: 22.57 min
Service Time: 196.7 min
Total Time: 219.27 min


In [31]:
# pip install pulp
import pulp
import math
import random

# -----------------------------
# PARAMETERS
# -----------------------------
N = 30
customers = list(range(1, N+1))
depot = 0
nodes = [depot] + customers

m = 1
vehicles = [1]

speed = 25.23     # mph
H = 240           # shift minutes
BIG = 10000

# -----------------------------
# Package demand
# -----------------------------
q = {j:1 for j in customers}
Q = {1:999}

# -----------------------------
# MOST service times
# -----------------------------
s = {}

for j in customers:
    service_type = random.choice(["house","company","apartment"])
    
    if service_type == "house":
        s[j] = 12.7
    elif service_type == "company":
        s[j] = 13.4
    else:
        s[j] = 24.6

# -----------------------------
# Coordinates
# customers located 5–7 miles from depot
# -----------------------------
coords = {0:(0,0)}

for j in customers:
    angle = 2*math.pi*j/N
    radius = random.uniform(5,7)
    
    coords[j] = (
        radius*math.cos(angle),
        radius*math.sin(angle)
    )

# -----------------------------
# Distance matrix
# -----------------------------
d = {}

for i in nodes:
    for j in nodes:
        if i != j:
            x1,y1 = coords[i]
            x2,y2 = coords[j]
            
            d[i,j] = math.sqrt(
                (x1-x2)**2 + (y1-y2)**2
            )

# -----------------------------
# MODEL
# -----------------------------
model = pulp.LpProblem(
    "MILP_VRP_13_Customers",
    pulp.LpMaximize
)

# route variable
x = pulp.LpVariable.dicts(
    "x",
    [(i,j) for i in nodes for j in nodes if i!=j],
    cat="Binary"
)

# customer served variable
y = pulp.LpVariable.dicts(
    "y",
    customers,
    cat="Binary"
)

# MTZ variables
u = pulp.LpVariable.dicts(
    "u",
    customers,
    lowBound=1,
    upBound=N,
    cat="Continuous"
)

# -----------------------------
# OBJECTIVE
# maximize deliveries first
# minimize distance second
# -----------------------------
model += (
    BIG * pulp.lpSum(y[j] for j in customers)
    -
    pulp.lpSum(
        d[i,j]*x[i,j]
        for i in nodes
        for j in nodes
        if i!=j
    )
)

# -----------------------------
# CUSTOMER VISIT CONSTRAINT
# optional delivery
# -----------------------------
for j in customers:
    
    model += (
        pulp.lpSum(
            x[i,j]
            for i in nodes if i!=j
        ) == y[j]
    )
    
    model += (
        pulp.lpSum(
            x[j,k]
            for k in nodes if j!=k
        ) == y[j]
    )

# -----------------------------
# DEPOT START
# -----------------------------
model += (
    pulp.lpSum(
        x[0,j] for j in customers
    ) <= 1
)

# -----------------------------
# DEPOT RETURN
# -----------------------------
model += (
    pulp.lpSum(
        x[i,0] for i in customers
    ) <= 1
)

# -----------------------------
# FLOW CONSERVATION
# -----------------------------
for h in customers:
    
    model += (
        pulp.lpSum(
            x[i,h]
            for i in nodes if i!=h
        )
        ==
        pulp.lpSum(
            x[h,j]
            for j in nodes if j!=h
        )
    )

# -----------------------------
# CAPACITY CONSTRAINT
# -----------------------------
model += (
    pulp.lpSum(
        q[j]*y[j]
        for j in customers
    )
    <= Q[1]
)

# -----------------------------
# TRAVEL TIME
# -----------------------------
travel_time = pulp.lpSum(
    (d[i,j]/speed)*60*x[i,j]
    for i in nodes
    for j in nodes
    if i!=j
)

# -----------------------------
# SERVICE TIME
# -----------------------------
service_time = pulp.lpSum(
    s[j]*y[j]
    for j in customers
)

# -----------------------------
# SHIFT LIMIT
# -----------------------------
model += (
    travel_time + service_time <= H
)

# -----------------------------
# MTZ SUBTOUR ELIMINATION
# -----------------------------
for i in customers:
    for j in customers:
        if i != j:
            model += (
                u[i] - u[j] + N*x[i,j]
                <= N-1
            )

# -----------------------------
# SOLVE
# -----------------------------
solver = pulp.PULP_CBC_CMD(msg=True)
model.solve(solver)

print("Status:", pulp.LpStatus[model.status])

if pulp.LpStatus[model.status] not in ["Optimal","Feasible"]:
    print("No feasible route found")
    
else:
    
    served = [
        j for j in customers
        if pulp.value(y[j]) > 0.5
    ]
    
    missed = N - len(served)
    
    print("Completed Deliveries:", len(served))
    print("Missed Deliveries:", missed)
    print("Penalty Cost: $", round(missed*1.20,2))
    
    # -------------------------
    # Route reconstruction
    # -------------------------
    route = [0]
    current = 0
    
    while True:
        
        next_node = None
        
        for j in nodes:
            if current != j and (current,j) in x:
                
                if pulp.value(x[current,j]) > 0.5:
                    next_node = j
                    break
        
        if next_node is None:
            break
        
        route.append(next_node)
        
        if next_node == 0:
            break
        
        current = next_node
    
    print("Optimal Route:", route)
    
    # -------------------------
    # Distance
    # -------------------------
    total_distance = 0
    
    for i in range(len(route)-1):
        total_distance += d[
            route[i],
            route[i+1]
        ]
    
    print("Total Distance:",
          round(total_distance,2),
          "miles")
    
    # -------------------------
    # Travel time
    # -------------------------
    total_travel_time = (
        total_distance/speed
    )*60
    
    print("Travel Time:",
          round(total_travel_time,2),
          "min")
    
    # -------------------------
    # Service time
    # -------------------------
    total_service_time = sum(
        s[j] for j in served
    )
    
    print("Service Time:",
          round(total_service_time,2),
          "min")
    
    # -------------------------
    # Total elapsed time
    # -------------------------
    total_elapsed = (
        total_travel_time
        +
        total_service_time
    )
    
    print("Elapsed Time:",
          round(total_elapsed,2),
          "min")
    
    # -------------------------
    # Utilization
    # -------------------------
    utilization = (
        total_elapsed/H
    )*100
    
    print("Utilization:",
          round(utilization,2),
          "%")

Status: Optimal
Completed Deliveries: 12
Missed Deliveries: 18
Penalty Cost: $ 21.6
Optimal Route: [0, 19, 18, 17, 14, 13, 12, 10, 8, 7, 6, 5, 4, 0]
Total Distance: 32.57 miles
Travel Time: 77.45 min
Service Time: 155.9 min
Elapsed Time: 233.35 min
Utilization: 97.23 %


In [33]:
# pip install pulp

import pulp
import math
import random
import time

# -----------------------------
# PARAMETERS
# -----------------------------
N = 30
customers = list(range(1, N+1))
depot = 0
nodes = [depot] + customers

m = 1
vehicles = [2]

speed = 25.23
H = 240
BIG = 10000

# -----------------------------
# PACKAGE DEMAND
# -----------------------------
q = {j:1 for j in customers}
Q = {1:999}

# -----------------------------
# SERVICE TIMES
# -----------------------------
s = {}

for j in customers:
    
    service_type = random.choice(
        ["house","company","apartment"]
    )
    
    if service_type == "house":
        s[j] = 12.7
        
    elif service_type == "company":
        s[j] = 13.4
        
    else:
        s[j] = 24.6

# -----------------------------
# CUSTOMER COORDINATES
# (5–7 miles from depot)
# -----------------------------
coords = {0:(0,0)}

for j in customers:
    
    angle = 2*math.pi*j/N
    radius = random.uniform(5,7)
    
    coords[j] = (
        radius*math.cos(angle),
        radius*math.sin(angle)
    )

# -----------------------------
# DISTANCE MATRIX
# -----------------------------
d = {}

for i in nodes:
    for j in nodes:
        
        if i != j:
            
            x1,y1 = coords[i]
            x2,y2 = coords[j]
            
            d[i,j] = math.sqrt(
                (x1-x2)**2 +
                (y1-y2)**2
            )

# -----------------------------
# MODEL
# -----------------------------
model = pulp.LpProblem(
    "MILP_VRP",
    pulp.LpMaximize
)

# -----------------------------
# DECISION VARIABLES
# -----------------------------
x = pulp.LpVariable.dicts(
    "x",
    [(i,j) for i in nodes for j in nodes if i!=j],
    cat="Binary"
)

y = pulp.LpVariable.dicts(
    "y",
    customers,
    cat="Binary"
)

u = pulp.LpVariable.dicts(
    "u",
    customers,
    lowBound=1,
    upBound=N,
    cat="Continuous"
)

# -----------------------------
# OBJECTIVE
# maximize deliveries
# minimize distance
# -----------------------------
model += (
    BIG * pulp.lpSum(
        y[j] for j in customers
    )
    -
    pulp.lpSum(
        d[i,j]*x[i,j]
        for i in nodes
        for j in nodes
        if i != j
    )
)

# -----------------------------
# CUSTOMER VISIT CONSTRAINT
# -----------------------------
for j in customers:
    
    model += (
        pulp.lpSum(
            x[i,j]
            for i in nodes
            if i != j
        ) == y[j]
    )
    
    model += (
        pulp.lpSum(
            x[j,k]
            for k in nodes
            if j != k
        ) == y[j]
    )

# -----------------------------
# DEPOT START
# -----------------------------
model += (
    pulp.lpSum(
        x[0,j]
        for j in customers
    ) <= 1
)

# -----------------------------
# DEPOT RETURN
# -----------------------------
model += (
    pulp.lpSum(
        x[i,0]
        for i in customers
    ) <= 1
)

# -----------------------------
# FLOW CONSERVATION
# -----------------------------
for h in customers:
    
    model += (
        pulp.lpSum(
            x[i,h]
            for i in nodes
            if i != h
        )
        ==
        pulp.lpSum(
            x[h,j]
            for j in nodes
            if j != h
        )
    )

# -----------------------------
# CAPACITY CONSTRAINT
# -----------------------------
model += (
    pulp.lpSum(
        q[j]*y[j]
        for j in customers
    )
    <= Q[1]
)

# -----------------------------
# TRAVEL TIME
# -----------------------------
travel_time = pulp.lpSum(
    (d[i,j]/speed)*60*x[i,j]
    for i in nodes
    for j in nodes
    if i != j
)

# -----------------------------
# SERVICE TIME
# -----------------------------
service_time = pulp.lpSum(
    s[j]*y[j]
    for j in customers
)

# -----------------------------
# SHIFT LIMIT
# -----------------------------
model += (
    travel_time + service_time <= H
)

# -----------------------------
# MTZ SUBTOUR ELIMINATION
# -----------------------------
for i in customers:
    for j in customers:
        
        if i != j:
            
            model += (
                u[i]
                -
                u[j]
                +
                N*x[i,j]
                <= N-1
            )

# -----------------------------
# SOLVE + COMPUTATIONAL TIMER
# -----------------------------
start_time = time.time()

solver = pulp.PULP_CBC_CMD(msg=True)
model.solve(solver)

end_time = time.time()

solver_runtime = end_time - start_time

print("\nStatus:",
      pulp.LpStatus[model.status])

print("\nComputational Runtime:")
print("Seconds:",
      round(solver_runtime,2))

print("Minutes:",
      round(solver_runtime/60,2))

print("Hours:",
      round(solver_runtime/3600,4))

# -----------------------------
# OUTPUT RESULTS
# -----------------------------
if pulp.LpStatus[model.status] not in ["Optimal","Feasible"]:
    
    print("No feasible route found")

else:
    
    served = [
        j for j in customers
        if pulp.value(y[j]) > 0.5
    ]
    
    missed = N - len(served)
    
    print("\nCompleted Deliveries:",
          len(served))
    
    print("Missed Deliveries:",
          missed)
    
    print("Penalty Cost: $",
          round(missed*1.20,2))

    # -------------------------
    # Route reconstruction
    # -------------------------
    route = [0]
    current = 0

    while True:
        
        next_node = None
        
        for j in nodes:
            
            if current != j and (current,j) in x:
                
                if pulp.value(
                    x[current,j]
                ) > 0.5:
                    
                    next_node = j
                    break
        
        if next_node is None:
            break
        
        route.append(next_node)
        
        if next_node == 0:
            break
        
        current = next_node

    print("\nOptimal Route:",
          route)

    # -------------------------
    # Total distance
    # -------------------------
    total_distance = 0

    for i in range(len(route)-1):
        
        total_distance += d[
            route[i],
            route[i+1]
        ]

    print("Total Distance:",
          round(total_distance,2),
          "miles")

    # -------------------------
    # Travel time
    # -------------------------
    total_travel_time = (
        total_distance/speed
    )*60

    print("Travel Time:",
          round(total_travel_time,2),
          "min")

    # -------------------------
    # Service time
    # -------------------------
    total_service_time = sum(
        s[j]
        for j in served
    )

    print("Service Time:",
          round(total_service_time,2),
          "min")

    # -------------------------
    # Total elapsed time
    # -------------------------
    total_elapsed = (
        total_travel_time
        +
        total_service_time
    )

    print("Elapsed Time:",
          round(total_elapsed,2),
          "min")

    # -------------------------
    # Utilization
    # -------------------------
    utilization = (
        total_elapsed/H
    )*100

    print("Utilization:",
          round(utilization,2),
          "%")


Status: Optimal

Computational Runtime:
Seconds: 6867.15
Minutes: 114.45
Hours: 1.9075

Completed Deliveries: 12
Missed Deliveries: 18
Penalty Cost: $ 21.6

Optimal Route: [0, 12, 11, 9, 8, 7, 6, 4, 3, 2, 1, 30, 29, 0]
Total Distance: 28.09 miles
Travel Time: 66.8 min
Service Time: 169.2 min
Elapsed Time: 236.0 min
Utilization: 98.33 %


In [35]:
import math
import random
import time
import pandas as pd
import copy

# =====================================================
# GLOBAL PARAMETERS
# =====================================================
N = 30
customers = list(range(1, N + 1))
depot = 0
nodes = [depot] + customers

speed = 25.23
H = 240
PENALTY_PER_MISSED = 1.20

ITERATIONS = 3000
REMOVE_RATE = 0.25
random.seed(42)

# =====================================================
# DEMAND AND SERVICE TIMES
# =====================================================
q = {j: 1 for j in customers}

s = {}
customer_type = {}

for j in customers:
    service_type = random.choice(["house", "company", "apartment"])
    customer_type[j] = service_type

    if service_type == "house":
        s[j] = 12.7
    elif service_type == "company":
        s[j] = 13.4
    else:
        s[j] = 24.6

# =====================================================
# CUSTOMER COORDINATES: 5–7 MILES FROM DEPOT
# =====================================================
coords = {0: (0, 0)}

for j in customers:
    angle = 2 * math.pi * j / N
    radius = random.uniform(5, 7)

    coords[j] = (
        radius * math.cos(angle),
        radius * math.sin(angle)
    )

# =====================================================
# DISTANCE MATRIX
# =====================================================
d = {}

for i in nodes:
    for j in nodes:
        if i != j:
            x1, y1 = coords[i]
            x2, y2 = coords[j]
            d[i, j] = math.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)


# =====================================================
# BASIC ROUTE FUNCTIONS
# =====================================================
def route_distance(route):
    return sum(d[route[i], route[i + 1]] for i in range(len(route) - 1))


def route_travel_time(route):
    return route_distance(route) / speed * 60


def route_service_time(route):
    return sum(s[j] for j in route if j != depot)


def route_elapsed_time(route):
    return route_travel_time(route) + route_service_time(route)


def is_route_feasible(route):
    return route_elapsed_time(route) <= H


def solution_distance(solution):
    return sum(route_distance(route) for route in solution)


def solution_elapsed(solution):
    return sum(route_elapsed_time(route) for route in solution)


def served_customers(solution):
    served = []
    for route in solution:
        served.extend([j for j in route if j != depot])
    return served


def missed_customers(solution):
    served = set(served_customers(solution))
    return [j for j in customers if j not in served]


def objective(solution):
    completed = len(served_customers(solution))
    missed = N - completed
    total_distance = solution_distance(solution)

    # Large reward for completed deliveries, small penalty for distance and missed deliveries
    return 10000 * completed - total_distance - missed * PENALTY_PER_MISSED


# =====================================================
# INITIAL SOLUTION: GREEDY NEAREST INSERTION
# =====================================================
def create_initial_solution(m):
    routes = [[depot, depot] for _ in range(m)]
    unassigned = customers.copy()

    # Sort by service time, shorter tasks first
    unassigned.sort(key=lambda j: s[j])

    for customer in unassigned[:]:

        best_route_index = None
        best_position = None
        best_increase = float("inf")

        for r_idx, route in enumerate(routes):
            for pos in range(1, len(route)):
                candidate_route = route[:pos] + [customer] + route[pos:]

                if is_route_feasible(candidate_route):
                    increase = route_distance(candidate_route) - route_distance(route)

                    if increase < best_increase:
                        best_increase = increase
                        best_route_index = r_idx
                        best_position = pos

        if best_route_index is not None:
            routes[best_route_index].insert(best_position, customer)

    return routes


# =====================================================
# DESTROY OPERATORS
# =====================================================
def random_removal(solution, remove_count):
    new_solution = copy.deepcopy(solution)
    all_served = served_customers(new_solution)

    if len(all_served) == 0:
        return new_solution, []

    remove_count = min(remove_count, len(all_served))
    removed = random.sample(all_served, remove_count)

    for route in new_solution:
        route[:] = [node for node in route if node not in removed]

        if route[0] != depot:
            route.insert(0, depot)
        if route[-1] != depot:
            route.append(depot)

    return new_solution, removed


def worst_removal(solution, remove_count):
    new_solution = copy.deepcopy(solution)

    savings = []

    for route in new_solution:
        for idx in range(1, len(route) - 1):
            customer = route[idx]
            before = route[idx - 1]
            after = route[idx + 1]

            saving = d[before, customer] + d[customer, after] - d[before, after]
            savings.append((saving, customer))

    savings.sort(reverse=True)

    removed = [customer for _, customer in savings[:remove_count]]

    for route in new_solution:
        route[:] = [node for node in route if node not in removed]

        if route[0] != depot:
            route.insert(0, depot)
        if route[-1] != depot:
            route.append(depot)

    return new_solution, removed


def related_removal(solution, remove_count):
    new_solution = copy.deepcopy(solution)
    all_served = served_customers(new_solution)

    if not all_served:
        return new_solution, []

    seed_customer = random.choice(all_served)

    related = sorted(
        all_served,
        key=lambda j: d[seed_customer, j] if j != seed_customer else -1
    )

    removed = related[:remove_count]

    for route in new_solution:
        route[:] = [node for node in route if node not in removed]

        if route[0] != depot:
            route.insert(0, depot)
        if route[-1] != depot:
            route.append(depot)

    return new_solution, removed


# =====================================================
# REPAIR OPERATORS
# =====================================================
def greedy_repair(solution, removed):
    new_solution = copy.deepcopy(solution)

    for customer in removed:

        best_route_index = None
        best_position = None
        best_increase = float("inf")

        for r_idx, route in enumerate(new_solution):
            for pos in range(1, len(route)):
                candidate_route = route[:pos] + [customer] + route[pos:]

                if is_route_feasible(candidate_route):
                    increase = route_distance(candidate_route) - route_distance(route)

                    if increase < best_increase:
                        best_increase = increase
                        best_route_index = r_idx
                        best_position = pos

        if best_route_index is not None:
            new_solution[best_route_index].insert(best_position, customer)

    return new_solution


def regret_repair(solution, removed):
    new_solution = copy.deepcopy(solution)
    remaining = removed.copy()

    while remaining:

        regret_list = []

        for customer in remaining:
            insertion_costs = []

            for r_idx, route in enumerate(new_solution):
                for pos in range(1, len(route)):
                    candidate_route = route[:pos] + [customer] + route[pos:]

                    if is_route_feasible(candidate_route):
                        increase = route_distance(candidate_route) - route_distance(route)
                        insertion_costs.append((increase, r_idx, pos))

            insertion_costs.sort()

            if len(insertion_costs) >= 2:
                regret = insertion_costs[1][0] - insertion_costs[0][0]
                regret_list.append((regret, customer, insertion_costs[0]))
            elif len(insertion_costs) == 1:
                regret_list.append((9999, customer, insertion_costs[0]))

        if not regret_list:
            break

        regret_list.sort(reverse=True)
        _, selected_customer, best_insert = regret_list[0]

        _, best_route_index, best_position = best_insert

        new_solution[best_route_index].insert(best_position, selected_customer)
        remaining.remove(selected_customer)

    return new_solution


# =====================================================
# INSERT MISSED CUSTOMERS IF POSSIBLE
# =====================================================
def insert_missed_customers(solution):
    new_solution = copy.deepcopy(solution)
    missed = missed_customers(new_solution)

    for customer in missed:

        best_route_index = None
        best_position = None
        best_increase = float("inf")

        for r_idx, route in enumerate(new_solution):
            for pos in range(1, len(route)):
                candidate_route = route[:pos] + [customer] + route[pos:]

                if is_route_feasible(candidate_route):
                    increase = route_distance(candidate_route) - route_distance(route)

                    if increase < best_increase:
                        best_increase = increase
                        best_route_index = r_idx
                        best_position = pos

        if best_route_index is not None:
            new_solution[best_route_index].insert(best_position, customer)

    return new_solution


# =====================================================
# ALNS MAIN FUNCTION
# =====================================================
def solve_alns(m):

    start_time = time.time()

    current_solution = create_initial_solution(m)
    current_solution = insert_missed_customers(current_solution)

    best_solution = copy.deepcopy(current_solution)

    destroy_operators = [random_removal, worst_removal, related_removal]
    repair_operators = [greedy_repair, regret_repair]

    destroy_weights = [1.0, 1.0, 1.0]
    repair_weights = [1.0, 1.0]

    temperature = 100
    cooling_rate = 0.995

    for iteration in range(ITERATIONS):

        remove_count = max(1, int(REMOVE_RATE * max(1, len(served_customers(current_solution)))))

        destroy_op = random.choices(
            destroy_operators,
            weights=destroy_weights,
            k=1
        )[0]

        repair_op = random.choices(
            repair_operators,
            weights=repair_weights,
            k=1
        )[0]

        partial_solution, removed = destroy_op(current_solution, remove_count)
        candidate_solution = repair_op(partial_solution, removed)

        # Try to insert previously missed customers too
        candidate_solution = insert_missed_customers(candidate_solution)

        current_obj = objective(current_solution)
        candidate_obj = objective(candidate_solution)
        best_obj = objective(best_solution)

        accept = False

        if candidate_obj > current_obj:
            accept = True
        else:
            probability = math.exp((candidate_obj - current_obj) / max(temperature, 1e-9))
            if random.random() < probability:
                accept = True

        if accept:
            current_solution = candidate_solution

        if candidate_obj > best_obj:
            best_solution = copy.deepcopy(candidate_solution)

            d_index = destroy_operators.index(destroy_op)
            r_index = repair_operators.index(repair_op)

            destroy_weights[d_index] += 0.2
            repair_weights[r_index] += 0.2

        temperature *= cooling_rate

    runtime = time.time() - start_time

    return best_solution, runtime


# =====================================================
# RUN ALNS FOR 1 TO 5 VEHICLES
# =====================================================
summary_all = []
vehicle_all = []

for m in [1, 2, 3, 4, 5]:

    print(f"\nRunning ALNS for {m} vehicle(s)...")

    solution, runtime = solve_alns(m)

    completed = len(served_customers(solution))
    missed = N - completed
    penalty = missed * PENALTY_PER_MISSED

    total_distance = solution_distance(solution)
    total_travel = sum(route_travel_time(route) for route in solution)
    total_service = sum(route_service_time(route) for route in solution)
    total_elapsed = sum(route_elapsed_time(route) for route in solution)

    summary_all.append({
        "Vehicles": m,
        "Completed Deliveries": completed,
        "Missed Deliveries": missed,
        "Penalty Cost": round(penalty, 2),
        "Total Fleet Distance miles": round(total_distance, 2),
        "Total Fleet Travel min": round(total_travel, 2),
        "Total Fleet Service min": round(total_service, 2),
        "Total Fleet Elapsed min": round(total_elapsed, 2),
        "Runtime Seconds": round(runtime, 2),
        "Runtime Minutes": round(runtime / 60, 2),
        "Runtime Hours": round(runtime / 3600, 4)
    })

    for vehicle_id, route in enumerate(solution, start=1):

        route_customers = [j for j in route if j != depot]

        distance = route_distance(route)
        travel = route_travel_time(route)
        service = route_service_time(route)
        elapsed = route_elapsed_time(route)
        utilization = elapsed / H * 100

        vehicle_all.append({
            "Scenario Vehicles": m,
            "Vehicle": vehicle_id,
            "Route": route,
            "Completed": len(route_customers),
            "Distance miles": round(distance, 2),
            "Travel min": round(travel, 2),
            "Service min": round(service, 2),
            "Elapsed min": round(elapsed, 2),
            "Utilization %": round(utilization, 2)
        })

# =====================================================
# CREATE RESULT TABLES
# =====================================================
summary_df = pd.DataFrame(summary_all)
vehicle_df = pd.DataFrame(vehicle_all)

print("\n==============================")
print("ALNS SUMMARY RESULT")
print("==============================")
print(summary_df)

print("\n==============================")
print("ALNS EACH VEHICLE RESULT")
print("==============================")
print(vehicle_df)

# =====================================================
# SAVE OUTPUT FILES
# =====================================================
summary_df.to_csv("ALNS_summary_1_to_5_vehicles.csv", index=False)
vehicle_df.to_csv("ALNS_each_vehicle_1_to_5_vehicles.csv", index=False)

summary_df.to_excel("ALNS_summary_1_to_5_vehicles.xlsx", index=False)
vehicle_df.to_excel("ALNS_each_vehicle_1_to_5_vehicles.xlsx", index=False)

print("\nFiles saved:")
print("ALNS_summary_1_to_5_vehicles.csv")
print("ALNS_each_vehicle_1_to_5_vehicles.csv")
print("ALNS_summary_1_to_5_vehicles.xlsx")
print("ALNS_each_vehicle_1_to_5_vehicles.xlsx")


Running ALNS for 1 vehicle(s)...


KeyError: (0, 0)

In [39]:
import math
import random
import time
import pandas as pd
import copy

# =====================================================
# PARAMETERS
# =====================================================
N = 30
customers = list(range(1, N + 1))
depot = 0
nodes = [depot] + customers

speed = 25.23
H = 240
PENALTY_PER_MISSED = 1.20

ITERATIONS = 3000
REMOVE_RATE = 0.25

random.seed(42)

# =====================================================
# DEMAND
# =====================================================
q = {j: 1 for j in customers}

# =====================================================
# SERVICE TIMES
# =====================================================
s = {}
customer_type = {}

for j in customers:

    service_type = random.choice(
        ["house", "company", "apartment"]
    )

    customer_type[j] = service_type

    if service_type == "house":
        s[j] = 12.7

    elif service_type == "company":
        s[j] = 13.4

    else:
        s[j] = 24.6

# =====================================================
# CUSTOMER COORDINATES
# =====================================================
coords = {0: (0, 0)}

for j in customers:

    angle = 2 * math.pi * j / N
    radius = random.uniform(5, 7)

    coords[j] = (
        radius * math.cos(angle),
        radius * math.sin(angle)
    )

# =====================================================
# DISTANCE MATRIX
# =====================================================
d = {}

for i in nodes:
    for j in nodes:

        if i != j:

            x1, y1 = coords[i]
            x2, y2 = coords[j]

            d[i, j] = math.sqrt(
                (x1 - x2) ** 2 +
                (y1 - y2) ** 2
            )

# =====================================================
# ROUTE FUNCTIONS
# =====================================================
def route_distance(route):

    total = 0

    for i in range(len(route)-1):

        a = route[i]
        b = route[i+1]

        # ignore depot-depot
        if a == b:
            continue

        total += d[a, b]

    return total


def route_travel_time(route):
    return (route_distance(route)/speed)*60


def route_service_time(route):
    return sum(
        s[j]
        for j in route
        if j != depot
    )


def route_elapsed_time(route):
    return (
        route_travel_time(route)
        +
        route_service_time(route)
    )


def is_route_feasible(route):
    return route_elapsed_time(route) <= H


def solution_distance(solution):
    return sum(
        route_distance(route)
        for route in solution
    )


def solution_elapsed(solution):
    return sum(
        route_elapsed_time(route)
        for route in solution
    )


def served_customers(solution):

    served = []

    for route in solution:
        served.extend(
            [j for j in route if j != depot]
        )

    return served


def missed_customers(solution):

    served = set(
        served_customers(solution)
    )

    return [
        j for j in customers
        if j not in served
    ]


def objective(solution):

    completed = len(
        served_customers(solution)
    )

    missed = N - completed

    total_distance = solution_distance(solution)

    return (
        10000*completed
        -
        total_distance
        -
        missed*PENALTY_PER_MISSED
    )

# =====================================================
# INITIAL SOLUTION
# =====================================================
def create_initial_solution(m):

    routes = [
        [depot, depot]
        for _ in range(m)
    ]

    unassigned = customers.copy()

    unassigned.sort(
        key=lambda j: s[j]
    )

    for customer in unassigned[:]:

        best_route_index = None
        best_position = None
        best_increase = float("inf")

        for r_idx, route in enumerate(routes):

            for pos in range(1, len(route)):

                candidate_route = (
                    route[:pos]
                    +
                    [customer]
                    +
                    route[pos:]
                )

                if is_route_feasible(candidate_route):

                    increase = (
                        route_distance(candidate_route)
                        -
                        route_distance(route)
                    )

                    if increase < best_increase:
                        best_increase = increase
                        best_route_index = r_idx
                        best_position = pos

        if best_route_index is not None:

            routes[best_route_index].insert(
                best_position,
                customer
            )

    return routes

# =====================================================
# DESTROY OPERATORS
# =====================================================
def random_removal(solution, remove_count):

    new_solution = copy.deepcopy(solution)

    all_served = served_customers(new_solution)

    if len(all_served) == 0:
        return new_solution, []

    remove_count = min(
        remove_count,
        len(all_served)
    )

    removed = random.sample(
        all_served,
        remove_count
    )

    for route in new_solution:

        route[:] = [
            node for node in route
            if node not in removed
        ]

        if route[0] != depot:
            route.insert(0, depot)

        if route[-1] != depot:
            route.append(depot)

    return new_solution, removed


def worst_removal(solution, remove_count):

    new_solution = copy.deepcopy(solution)
    savings = []

    for route in new_solution:

        # skip empty routes
        if len(route) <= 2:
            continue

        for idx in range(1, len(route)-1):

            customer = route[idx]
            before = route[idx-1]
            after = route[idx+1]

            # safe distance calculations
            dist_before_customer = 0
            dist_customer_after = 0
            dist_before_after = 0

            if before != customer:
                dist_before_customer = d[before, customer]

            if customer != after:
                dist_customer_after = d[customer, after]

            if before != after:
                dist_before_after = d[before, after]

            saving = (
                dist_before_customer
                +
                dist_customer_after
                -
                dist_before_after
            )

            savings.append(
                (saving, customer)
            )

    # if no savings found
    if len(savings) == 0:
        return new_solution, []

    savings.sort(reverse=True)

    removed = [
        customer
        for _, customer in savings[:remove_count]
    ]

    for route in new_solution:

        route[:] = [
            node for node in route
            if node not in removed
        ]

        # maintain depot structure
        if len(route) == 0:
            route.extend([0,0])

        elif route[0] != 0:
            route.insert(0,0)

        elif route[-1] != 0:
            route.append(0)

    return new_solution, removed

# =====================================================
# REPAIR OPERATORS
# =====================================================
def greedy_repair(solution, removed):

    new_solution = copy.deepcopy(solution)

    for customer in removed:

        best_route = None
        best_position = None
        best_increase = float("inf")

        for r_idx, route in enumerate(new_solution):

            for pos in range(1, len(route)):

                candidate_route = (
                    route[:pos]
                    +
                    [customer]
                    +
                    route[pos:]
                )

                if is_route_feasible(candidate_route):

                    increase = (
                        route_distance(candidate_route)
                        -
                        route_distance(route)
                    )

                    if increase < best_increase:
                        best_increase = increase
                        best_route = r_idx
                        best_position = pos

        if best_route is not None:
            new_solution[best_route].insert(
                best_position,
                customer
            )

    return new_solution

# =====================================================
# ALNS MAIN
# =====================================================
def solve_alns(m):

    start_time = time.time()

    current_solution = create_initial_solution(m)
    best_solution = copy.deepcopy(current_solution)

    destroy_ops = [
        random_removal,
        worst_removal,
        related_removal
    ]

    temperature = 100
    cooling_rate = 0.995

    for iteration in range(ITERATIONS):

        remove_count = max(
            1,
            int(
                REMOVE_RATE
                *
                max(
                    1,
                    len(
                        served_customers(
                            current_solution
                        )
                    )
                )
            )
        )

        destroy_op = random.choice(
            destroy_ops
        )

        partial_solution, removed = destroy_op(
            current_solution,
            remove_count
        )

        candidate_solution = greedy_repair(
            partial_solution,
            removed
        )

        current_obj = objective(
            current_solution
        )

        candidate_obj = objective(
            candidate_solution
        )

        best_obj = objective(
            best_solution
        )

        accept = False

        if candidate_obj > current_obj:
            accept = True

        else:
            probability = math.exp(
                (candidate_obj-current_obj)
                /
                max(temperature,1e-9)
            )

            if random.random() < probability:
                accept = True

        if accept:
            current_solution = candidate_solution

        if candidate_obj > best_obj:
            best_solution = copy.deepcopy(
                candidate_solution
            )

        temperature *= cooling_rate

    runtime = time.time() - start_time

    return best_solution, runtime

# =====================================================
# RUN SCENARIOS 1 TO 5 VEHICLES
# =====================================================
summary_all = []
vehicle_all = []

for m in [1,2,3,4,5]:

    print(f"\nRunning ALNS for {m} vehicles...")

    solution, runtime = solve_alns(m)

    completed = len(
        served_customers(solution)
    )

    missed = N - completed

    penalty = missed * PENALTY_PER_MISSED

    total_distance = solution_distance(solution)
    total_travel = solution_distance(solution)/speed*60
    total_service = sum(
        route_service_time(route)
        for route in solution
    )

    total_elapsed = total_travel + total_service

    summary_all.append({
        "Vehicles":m,
        "Completed Deliveries":completed,
        "Missed Deliveries":missed,
        "Penalty Cost":round(penalty,2),
        "Total Distance":round(total_distance,2),
        "Total Travel Time":round(total_travel,2),
        "Total Service Time":round(total_service,2),
        "Total Elapsed Time":round(total_elapsed,2),
        "Runtime Seconds":round(runtime,2),
        "Runtime Minutes":round(runtime/60,2)
    })

    for v_id, route in enumerate(solution,start=1):

        route_customers = [
            j for j in route
            if j != depot
        ]

        distance = route_distance(route)
        travel = route_travel_time(route)
        service = route_service_time(route)
        elapsed = route_elapsed_time(route)

        utilization = elapsed/H*100

        vehicle_all.append({
            "Scenario Vehicles":m,
            "Vehicle":v_id,
            "Route":route,
            "Completed":len(route_customers),
            "Distance":round(distance,2),
            "Travel Time":round(travel,2),
            "Service Time":round(service,2),
            "Elapsed Time":round(elapsed,2),
            "Utilization %":round(utilization,2)
        })

# =====================================================
# RESULT TABLES
# =====================================================
summary_df = pd.DataFrame(summary_all)
vehicle_df = pd.DataFrame(vehicle_all)

print("\n==============================")
print("ALNS SUMMARY")
print("==============================")
print(summary_df)

print("\n==============================")
print("ALNS VEHICLE DETAILS")
print("==============================")
print(vehicle_df)

summary_df.to_excel(
    "ALNS_summary.xlsx",
    index=False
)

vehicle_df.to_excel(
    "ALNS_vehicle_details.xlsx",
    index=False
)

print("\nFiles saved:")
print("ALNS_summary.xlsx")
print("ALNS_vehicle_details.xlsx")


Running ALNS for 1 vehicles...

Running ALNS for 2 vehicles...

Running ALNS for 3 vehicles...

Running ALNS for 4 vehicles...

Running ALNS for 5 vehicles...

ALNS SUMMARY
   Vehicles  Completed Deliveries  Missed Deliveries  Penalty Cost  \
0         1                    11                 19          22.8   
1         2                    19                 11          13.2   
2         3                    26                  4           4.8   
3         4                    30                  0           0.0   
4         5                    30                  0           0.0   

   Total Distance  Total Travel Time  Total Service Time  Total Elapsed Time  \
0           34.95              83.11               139.7              222.81   
1           52.08             123.84               278.4              402.24   
2           65.68             156.19               450.6              606.79   
3           81.71             194.32               549.0              743.32   
4    